# Checklist Engine TOR

Notebook khusus evaluasi TOR Section I (I.A–I.T). Input hanya dibaca dari `input/TOR/*.json`. Kriteria I.M–I.R menggunakan hook `LLM_EVALUATOR` eksternal.

## Struktur file

```text
ROOT/
├── input/TOR/              # JSON hasil parsing TOR
├── rules/rules_config.json
└── hasil_cek/TOR/          # Laporan JSON dan Markdown
```

Letakkan file JSON TOR pada `input/TOR/`.

## 1. Konfigurasi

In [1]:
from pathlib import Path
import json, re, hashlib, datetime, os, textwrap, math, traceback

PROJECT_DIR = Path.cwd()
INPUT_ROOT = PROJECT_DIR / "input"
RULES_PATH = PROJECT_DIR / "rules" / "rules_config.json"
OUT_DIR = PROJECT_DIR / "hasil_cek"

ROUTES = {
    "TOR": {"folder": INPUT_ROOT / "TOR", "section_code": "I", "label": "Kelengkapan TOR"},
}
EXTRA_SECTIONS = {}
TARGET_FILES = {"TOR": []}  # [] = semua .json di folder TOR

USE_SFT_MODEL = False
# Evaluasi LLM I.M.1-I.R.1 dilakukan pada notebook/model terpisah.
# Isi LLM_EVALUATOR pada cell integrasi jika endpoint/fungsi evaluator sudah tersedia.
LLM_EVALUATOR = None
USE_LLM_FOR_SECTION_IV = False  # Section IV tetap memerlukan validasi lintas dokumen

# Integrasi model CPT+SFT teman: isi path setelah model/adapter siap.
USE_SFT_MODEL = False
SFT_BASE_MODEL_ID = "Qwen/Qwen3-1.7B-Base"
SFT_MERGED_MODEL_PATH = None      # path model hasil merge/finetune, jika ada
SFT_ADAPTER_PATH = None           # path adapter LoRA/PEFT, jika ada
SFT_MODEL_VERSION = "checklist-cpt-sft-not-configured"
MAX_NEW_TOKENS = 500

# Rule deterministik menggunakan nama key pada JSON; bukan semantic retrieval.
PIPELINE_VERSION = "key-value-checklist-v2.1.0"
PROMPT_VERSION = "external-llm-integration-v1"
RULES_VERSION = "checklist-rules-v1"  # ubah saat rules direvisi; hash rules juga dicatat

print("Project directory:", PROJECT_DIR.resolve())
print("Rules file:", RULES_PATH.resolve())
print("SFT enabled:", USE_SFT_MODEL, "| SFT config:", SFT_ADAPTER_PATH or SFT_MERGED_MODEL_PATH or "belum dikonfigurasi")


Project directory: D:\OneDrive\AITF\Checklist engine
Rules file: D:\OneDrive\AITF\Checklist engine\rules\rules_config.json
SFT enabled: False | SFT config: belum dikonfigurasi


## 2. Load dan validasi rules

In [2]:
if not RULES_PATH.is_file():
    raise FileNotFoundError(
        f"Rules tidak ditemukan: {RULES_PATH.resolve()}. "
        "Letakkan rules_config.json pada folder rules/."
    )

rules_raw = RULES_PATH.read_bytes()
cfg = json.loads(rules_raw.decode("utf-8-sig"))
RULES_HASH = hashlib.sha256(rules_raw).hexdigest()[:16]
sections = cfg.get("screening_satu_format_dan_substansi")
if not isinstance(sections, list):
    raise ValueError("rules_config.json harus memiliki list 'screening_satu_format_dan_substansi'.")

rules_by_code = {}
for section in sections:
    section_code = section.get("section_code")
    if not section_code:
        raise ValueError(f"Section tanpa section_code: {section}")
    if section_code in rules_by_code:
        raise ValueError(f"section_code duplikat: {section_code}")
    if not isinstance(section.get("items"), list):
        raise ValueError(f"Section {section_code} tidak memiliki list 'items'.")
    rules_by_code[section_code] = section

for kind, route in ROUTES.items():
    sc = route["section_code"]
    if sc not in rules_by_code:
        raise KeyError(f"Section {sc} untuk {kind} tidak ditemukan pada rules_config.json.")
    if kind == "TOR":
        route["rules"] = {
            **rules_by_code[sc],
            "items": [
                item for item in rules_by_code[sc]["items"]
                if str(item.get("kriteria_kode", "")).strip().startswith("I.")
            ],
        }
    else:
        route["rules"] = rules_by_code[sc]
    print(f"{kind:7} -> Section {sc}: {route['rules'].get('section_title', '')} "
          f"({len(route['rules']['items'])} kriteria)")

for kind, extras in EXTRA_SECTIONS.items():
    for ex in extras:
        if ex["section_code"] not in rules_by_code:
            raise KeyError(
                f"Section {ex['section_code']} untuk {kind} tidak ditemukan pada rules_config.json."
            )
        
print("Rules version:", RULES_VERSION, "| SHA-256:", RULES_HASH)

TOR     -> Section I: APAKAH TOR SESUAI DENGAN FORMAT? (KELENGKAPAN) (24 kriteria)
Rules version: checklist-rules-v1 | SHA-256: afa5d3ed61129a0e


## 3. Membaca JSON dan mempertahankan lokasi sumber

Loader ini mendukung beragam struktur JSON dengan mencari field teks umum seperti `raw_text`, `text`, `content`, `paragraph`, dan `value`. Metadata umum seperti halaman, bagian, judul, ID blok, serta jalur JSON dipertahankan bila tersedia.

**Jika JSON-mu memiliki schema khusus**, periksa fungsi `extract_chunks_from_json()` dan tambahkan nama field yang sesuai. Kutipan hanya bisa berupa teks yang memang tersedia pada JSON; halaman PDF asli tidak dapat direkonstruksi jika metadata halaman tidak disimpan saat parsing.

In [3]:
TEXT_KEYS = ("raw_text", "chunk_text", "markdown", "text", "content", "paragraph", "body", "quote")
META_KEYS = {
    "page", "page_number", "page_no", "section", "section_title", "heading",
    "block_id", "chunk_id", "source_id", "document_id", "file_name", "table", "row"
}
ENTITY_KEYS = ("entity", "label", "field", "name", "key")
VALUE_KEYS = ("value", "normalized_value", "text_value")

def first_present(d, keys):
    for key in keys:
        if key in d and d[key] not in (None, ""):
            return d[key]
    return None

def split_source_sentences(text):
    """Memecah teks menjadi kalimat sambil mempertahankan kutipan verbatim.
    Ini heuristik; teks tabel/fragmen tidak dipaksa menjadi kalimat buatan.
    """
    text = str(text or "").strip()
    if not text:
        return []
    #parts = re.split(r"(?<=[.!?])\s+(?=(?:[A-Z0-9\"Ã¢â‚¬Å“Ã¢â‚¬Ëœ(]))", text) //memecah nilai panjang menjadi kalimat, tapi bisa memecah kutipan verbatim
    parts = re.split(
        r"(?<!\d\.)(?<=[.!?])\s+(?=(?:[A-Z0-9\"â€œâ€˜(]))",
        text
    )
    return [p.strip() for p in parts if p.strip()] or [text]

def extract_chunks_from_json(obj, source_file):
    chunks = []

    def add_chunk(text, quote, path, meta, context=""):
        raw_quote = str(quote).strip()
        if not raw_quote:
            return
        # Split paragraf menjadi unit kalimat untuk evidence yang lebih spesifik.
        # Field pendek/entity-value dipertahankan sebagai fragmen sumber.
        sentence_units = split_source_sentences(raw_quote) if len(raw_quote) > 180 else [raw_quote]
        for unit in sentence_units:
            display_text = (context + " | " if context else "") + str(text).strip()
            if len(sentence_units) > 1:
                display_text = (context + " | " if context else "") + unit
            chunks.append({
                "chunk_id": f"{Path(source_file).stem}:{len(chunks)+1}",
                "text": display_text,
                "quote": unit,
                "source_file": str(source_file),
                "json_path": path,
                "page": meta.get("page_number", meta.get("page_no", meta.get("page"))),
                "section": meta.get("section_title", meta.get("section")),
                "heading": meta.get("heading"),
                "block_id": meta.get("block_id"),
                "source_id": meta.get("source_id"),
                "metadata": dict(meta),
                "evidence_type": "sentence" if re.search(r"[.!?]$", unit) else "source_fragment",
            })

    def walk(node, path="$", inherited=None):
        inherited = dict(inherited or {})
        if isinstance(node, dict):
            for k in META_KEYS:
                if node.get(k) not in (None, ""):
                    inherited[k] = node[k]
            payload_key = next((k for k in TEXT_KEYS if isinstance(node.get(k), str) and node[k].strip()), None)
            entity = first_present(node, ENTITY_KEYS)
            context = " | ".join(f"{k}: {node[k]}" for k in
                ("entity", "label", "field", "name", "key", "type")
                if isinstance(node.get(k), (str, int, float)) and str(node[k]).strip())
            if payload_key:
                add_chunk(node[payload_key], node[payload_key], f"{path}.{payload_key}", inherited, context)
                for k, v in node.items():
                    if k != payload_key and isinstance(v, (dict, list)):
                        walk(v, f"{path}.{k}", inherited)
                return
            value = first_present(node, VALUE_KEYS)
            if entity is not None and value is not None and not isinstance(value, (dict, list)):
                add_chunk(f"{entity}: {value}", value, path, inherited, str(entity))
            for k, v in node.items():
                if isinstance(v, (dict, list)):
                    walk(v, f"{path}.{k}", inherited)
                elif k not in TEXT_KEYS and v is not None and isinstance(v, (str, int, float)):
                    val = str(v).strip()
                    if k not in META_KEYS and (isinstance(v, (bool, int, float)) or len(val) >= 8):
                        add_chunk(f"{k}: {val}", val, f"{path}.{k}", inherited, str(k))
        elif isinstance(node, list):
            for i, item in enumerate(node):
                walk(item, f"{path}[{i}]", inherited)
        elif isinstance(node, str) and node.strip():
            add_chunk(node, node, path, inherited)

    walk(obj)
    seen, unique = set(), []
    for c in chunks:
        key = (c["source_file"], c["json_path"], c["quote"])
        if key not in seen:
            seen.add(key)
            unique.append(c)
    for i, c in enumerate(unique, 1):
        c["chunk_id"] = f"{Path(source_file).stem}:{i}"
    return unique

def load_json_document(path):
    raw = json.loads(path.read_text(encoding="utf-8-sig"))
    return {"path": str(path), "raw": raw, "chunks": extract_chunks_from_json(raw, path)}

docs_by_type = {}
for kind, route in ROUTES.items():
    folder = Path(route["folder"])
    docs_by_type[kind] = {}
    if not folder.is_dir():
        print(f"SKIP [{kind}]: folder tidak ditemukan: {folder.resolve()}")
        continue
    names = TARGET_FILES.get(kind) or []
    paths = [folder / n for n in names if (folder / n).is_file()] if names else sorted(folder.glob("*.json"))
    if not paths:
        print(f"SKIP [{kind}]: tidak ada file .json di {folder.resolve()}")
        continue
    for path in paths:
        try:
            doc = load_json_document(path)
            if not doc["chunks"]:
                print(f"SKIP [{kind}] {path.name}: tidak ditemukan teks yang bisa dievaluasi")
                continue
            docs_by_type[kind][path.name] = doc
            print(f"LOADED [{kind}] {path.name}: {len(doc['chunks'])} chunks")
        except (OSError, UnicodeError, json.JSONDecodeError) as e:
            print(f"SKIP [{kind}] {path.name}: gagal membaca JSON: {e}")
print("Jumlah dokumen:", {k: len(v) for k, v in docs_by_type.items()})

LOADED [TOR] (PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx.json: 211 chunks
LOADED [TOR] (PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx_vlm.json: 211 chunks
LOADED [TOR] (PA.2027 PNBP) FAE.001 - TOR Pemantauan dan Evaluasi Kebijakan Teknologi Pemerintah Digital Rev180826.docx.json: 368 chunks
LOADED [TOR] 0. TOR_7437.BAH.U00 Pemantauan, Pengukuran, dan Penanganan Gangguan Frekuensi Radio Dalam Kota.json: 263 chunks
LOADED [TOR] 009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran.json: 366 chunks
LOADED [TOR] 009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran_vlm.json: 422 chunks
LOADED [TOR] 01. PAA - KAK Rancangan Revisi UU KIP No. 14 Tahun 2008.json: 287 chunks
LOADED [TOR] 011 - TOR 2027 OM Sistem Informasi Manajemen Layanan Publik Bidang Penyiaran Pemutakhiran.json: 668 chunks
LOADED [TOR] 011 - TOR 2027 OM Sistem Informasi Manajemen Layanan Publik Bidang Penyiaran Pemutakhiran_vlm.json: 684 chunks
LOAD

## 4. Deterministic key-value matching\n\nPemeriksaan format menggunakan nama key JSON dan nilai yang terkait. Tidak menggunakan embedding atau semantic retrieval.

In [4]:
# Semantic retrieval dihapus: struktur input bersifat key-value.

## 5. Integrasi evaluator LLM eksternal

In [5]:
# Model LLM dibangun/dimuat pada notebook terpisah.
# Hubungkan fungsi evaluator dari notebook tersebut ke LLM_EVALUATOR.
# Kontrak fungsi:
#   LLM_EVALUATOR(kind=..., doc_name=..., item=..., document=...)
# Return:
#   {"status": "LOLOS|TIDAK|MANUAL", "alasan": "...",
#    "evidence_quote": "<kutipan persis dari document['chunks']>",
#    "evidence_chunk_id": "<chunk_id>"}
llm_model = None
tokenizer = None
model_version = "external-llm-not-connected"
sft_model_ready = False

def run_external_llm_evaluator(kind, doc_name, item, document):
    if not callable(LLM_EVALUATOR):
        raise RuntimeError("LLM_EVALUATOR belum dihubungkan dari notebook LLM.")
    return LLM_EVALUATOR(
        kind=kind,
        doc_name=doc_name,
        item=item,
        document=document,
    )
model_version = "dummy-llm-v0 (BUKAN LLM ASLI)"

def dummy_llm_evaluator(kind, doc_name, item, document):
    return {
        "status": "MANUAL",
        "alasan": "DUMMY: LLM belum tersedia. Perlu review manual.",
        "evidence_quote": "",
        "evidence_chunk_id": None,
    }

LLM_EVALUATOR = dummy_llm_evaluator

## Kontrak integrasi evaluator LLM

Kriteria TOR I.M.1–I.R.1 diteruskan ke `LLM_EVALUATOR` dari notebook/model LLM terpisah. Jika evaluator belum dihubungkan, kriteria tersebut berstatus `MANUAL`.

In [6]:
OUTPUT_SCHEMA = {
    "status": "LOLOS | TIDAK | MANUAL",
    "alasan": "alasan spesifik terhadap kriteria dan evidence; jangan mengarang",
    "evidence_quote": "kutipan verbatim dari salah satu kandidat, atau string kosong",
    "evidence_chunk_id": "chunk_id kandidat, atau null"
}

def criterion_method(kind, item):
    """Routing metode sesuai kategori pemeriksaan yang diminta."""
    code = str(item.get("kriteria_kode", "")).strip().upper()
    match = re.match(r"^I\.([A-T])(?:[.\-]|$)", code)

    if kind == "TOR" and match:
        letter = match.group(1)
        if "M" <= letter <= "R":
            return "llm_integration"
        if "A" <= letter <= "L" or "S" <= letter <= "T":
            return "rule_based_key"

    if kind == "RAB":
        return "rule_based_key"
    return "manual"

def build_llm_integration_payload(kind, doc_name, item, document):
    """Payload standar untuk diteruskan ke evaluator LLM pada notebook terpisah."""
    return {
        "jenis_dokumen": kind,
        "nama_file": doc_name,
        "kriteria": item,
        "document_raw": document.get("raw", {}),
        "evidence_chunks": document.get("chunks", []),
        "output_schema": OUTPUT_SCHEMA,
    }


## 5. Rule engine deterministik berbasis key-value\n\nRule dibedakan berdasarkan jenis dokumen (`TOR`, `RAB`) dan kode kriteria. Pemeriksaan deterministik membaca nama key JSON serta memvalidasi nilai/flag terkait; tidak menggunakan pencarian kata semantik atau regex atas serialisasi JSON.

In [ ]:
import re

def _norm(s):
    return re.sub(r"\\s+", " ", str(s or "")).strip().casefold()

def _norm_key(s):
    return re.sub(r"[^a-z0-9]", "", str(s or "").casefold())

# Alias key JSON per kriteria. Alias dibuat spesifik per template/jenis dokumen.
# Pemeriksaan dilakukan terhadap KEY dan validitas nilai, bukan isi semantik.
KEY_RULES = {'TOR': {'I.A.1': ['I.A'],
         'I.B.1': ['I.B'],
         'I.C.1': ['I.C'],
         'I.D.1': ['I.D'],
         'I.E.1': ['I.E'],
         'I.F.1': ['I.F'],
         'I.G.1': ['I.G'],
         'I.H.1': ['I.H'],
         'I.I.1': ['I.I'],
         'I.J.1': ['I.J'],
         'I.K.1': ['I.K'],
         'I.L.1': ['I.L'],
         'I.M.1': ['I.M.1', 'I.M.2', 'I.M.3'],
         'I.M.2': ['I.M.2'],
         'I.M.3': ['I.M.3'],
         'I.N.1': ['I.N'],
         'I.O.1': ['I.O.1', 'I.O.2', 'I.O.3'],
         'I.O.2': ['I.O.2'],
         'I.O.3': ['I.O.3'],
         'I.P.1': ['I.P'],
         'I.Q.1': ['I.Q'],
         'I.R.1': ['I.R'],
         'I.S.1': ['I.S.1', 'I.S.2'],
         'I.T.1': ['I.T']}}

def walk_key_values(node, path="$"):
    """Yield (key, value, json_path) untuk seluruh pasangan key-value bersarang."""
    if isinstance(node, dict):
        for key, value in node.items():
            child_path = f"{path}.{key}"
            yield str(key), value, child_path
            yield from walk_key_values(value, child_path)
    elif isinstance(node, list):
        for i, value in enumerate(node):
            yield from walk_key_values(value, f"{path}[{i}]")

def _value_is_present(value):
    if value is None:
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (int, float)):
        return True
    if isinstance(value, bool):
        return True
    if isinstance(value, (dict, list)):
        return bool(value)
    return bool(str(value).strip())

def _key_rule_passes(rule_type, value, key):
    if rule_type == "nonempty":
        return _value_is_present(value)
    if rule_type == "positive_number":
        try:
            return float(value) > 0
        except (TypeError, ValueError):
            return False
    if rule_type == "nip":
        # NIP terstruktur dapat berupa string langsung atau objek hasil ekstraksi OCR.
        if isinstance(value, dict):
            return value.get("ada_label") is True or bool(re.search(r"\\b\\d{18}\\b", str(value)))
        return bool(re.search(r"\\b\\d{18}\\b", str(value or "")))
    if rule_type == "signature":
        # Nama pejabat harus tersedia dan metadata tanda tangan terdeteksi true.
        return _value_is_present(value)
    return False

def check_tor_fields(doc, criterion_code):
    raw = doc.get("raw", {})
    fields = raw.get("fields", {})

    field_codes = KEY_RULES.get("TOR", {}).get(criterion_code, [])

    if not field_codes:
        return {
            "status": "TIDAK",
            "evidence": [],
            "quote": "",
            "field_codes": field_codes,
            "note": f"Mapping field TOR tidak ditemukan untuk {criterion_code}"
        }

    evidence = []
    quotes = []
    missing = []
    is_llm_tor_criterion = bool(re.match(r"^I\.[M-R](?:[.\-]|$)", criterion_code))
    # I.S.1: kedua field (I.S.1 & I.S.2) harus present=true dengan value tidak kosong;
    # label_found tidak menjadi syarat pada format baru.
    presence_value_only = is_llm_tor_criterion or criterion_code.strip().upper() == "I.S.1"

    for field_code in field_codes:
        field = fields.get(field_code)

        if not isinstance(field, dict):
            missing.append(field_code)
            continue

        label = str(field.get("label", "")).strip()
        raw_value = field.get("value")
        value = str(raw_value).strip() if raw_value is not None else ""

        present = field.get("present") is True
        label_found = field.get("label_found") is True

        if not present or not value or (not presence_value_only and not label_found):
            missing.append(field_code)
            continue

        if label:
            evidence.append(f"key:{label}")

        quotes.append(value)

    if missing:
        return {
            "status": "TIDAK",
            "evidence": evidence,
            "quote": " ".join(quotes),
            "field_codes": field_codes,
            "note": (
                "Field TOR tidak memenuhi syarat "
                + ("present=true dan value tidak kosong: " if presence_value_only else "present=true, label_found=true, dan value tidak kosong: ")
                + ", ".join(missing)
            )
        }
    # di dalam check_tor_fields, sebelum return kondisi TIDAK:
    missing_detail = []
    for field_code in field_codes:
        field = fields.get(field_code)
        if not isinstance(field, dict):
            missing_detail.append({"field": field_code, "sebab": "tidak_ada"})
            continue
        value = str(field.get("value") or "").strip()
        if field.get("present") is not True:
            missing_detail.append({"field": field_code, "sebab": "tidak_terdeteksi"})
        elif not value:
            missing_detail.append({"field": field_code, "sebab": "nilai_kosong"})
        elif not presence_value_only and field.get("label_found") is not True:
            missing_detail.append({"field": field_code, "sebab": "label_tidak_ditemukan"})

    return {
        "status": "LOLOS",
        "evidence": list(dict.fromkeys(evidence)),
        "quote": "\n".join(quotes),
        "field_codes": field_codes,
        "note": "Field TOR ditemukan dan memenuhi pemeriksaan keberadaan.",
        "missing_detail": missing_detail
    }

def deterministic_key_check(kind, doc, item):
    code = str(item.get("kriteria_kode", "")).strip().upper()
    rule = KEY_RULES.get(kind, {}).get(code)
    if kind == "TOR":
        criterion_code = item.get("kriteria_kode", "")
        result = check_tor_fields(doc, criterion_code)

        matches = []

        if result["quote"]:
            matches.append({
                "quote": result["quote"],
                "chunk_id": None,
                "text": result["quote"],
                "json_path": f"$.fields.{criterion_code}"
            })

        return {
            "status": result["status"],
            "evidence": result["evidence"],
            "note": result["note"],
            "missing_detail": result.get("missing_detail", []),
        }, matches
    
    if not rule:
        return {"status": "MANUAL", "evidence": [], "note": "Key rule belum dikonfigurasi"}, []

    #entries = list(walk_key_values(doc.get("raw", {})))
    raw = doc.get("raw", {})

    metadata = raw.get("metadata", {})
    if not isinstance(metadata, (dict, list)):
        metadata = {}

    entries = list(walk_key_values(metadata, "$.metadata"))
    wanted = {_norm_key(k) for k in rule["keys"]}
    matches = [(key, value, path) for key, value, path in entries if _norm_key(key) in wanted]

    # Kriteria tanda tangan dan NIP membutuhkan flag verifikasi nested.
    # III.K.1 mensyaratkan nama dan jabatan, serta minimal satu flag tanda tangan bernilai true.
    if rule["type"] == "signature_all":
    # Ketiga field wajib ada dalam objek induk yang sama.
        signature_groups = {}

        for k, v, p in entries:
            normalized = _norm_key(k)
            if normalized in {"nama", "jabatan", "tandatangan", "tandatanganelektronik"}:
                parent_path = p.rsplit(".", 1)[0]
                signature_groups.setdefault(parent_path, {})[normalized] = v

        valid_signature_groups = {
            parent: fields
            for parent, fields in signature_groups.items()
            if _value_is_present(fields.get("nama"))
            and _value_is_present(fields.get("jabatan"))
            and (fields.get("tandatangan") is True or fields.get("tandatanganelektronik") is True)
        }

        passed = bool(valid_signature_groups)

        matches = [
            (k, v, p)
            for k, v, p in entries
            if _norm_key(k) in {"nama", "jabatan", "tandatangan", "tandatanganelektronik"}
            and p.rsplit(".", 1)[0] in valid_signature_groups
        ]


    elif rule["type"] == "signature":
        signature_flags = [v for k, v, p in entries
                           if _norm_key(k) == "detected" and ".tanda_tangan." in p]
        signature_flag = any(v is True for v in signature_flags)
        matches = [(k, v, p) for k, v, p in matches if _value_is_present(v)]
        passed = bool(matches) and signature_flag

    elif rule["type"] == "nip":
        # Ketiga field wajib ada dalam objek induk yang sama.
        nip_groups = {}

        for k, v, p in entries:
            normalized = _norm_key(k)
            if normalized in {"nip", "nipada", "nipvalid"}:
                parent_path = p.rsplit(".", 1)[0]
                nip_groups.setdefault(parent_path, {})[normalized] = v

        valid_nip_groups = {
            parent: fields
            for parent, fields in nip_groups.items()
            if isinstance(fields.get("nip"), str)
            and bool(fields["nip"].strip())
            and fields.get("nipada") is True
            and fields.get("nipvalid") is True
        }

        passed = bool(valid_nip_groups)

        matches = [
            (k, v, p)
            for k, v, p in entries
            if _norm_key(k) in {"nip", "nipada", "nipvalid"}
            and p.rsplit(".", 1)[0] in valid_nip_groups
        ]
    else:
        valid = [(k, v, p) for k, v, p in matches
                 if _key_rule_passes(rule["type"], v, k)]
        passed = bool(valid)
        matches = valid

    # Buat/ambil chunk evidence yang mengacu pada key-value sumber.
    chunks = doc.get("chunks", [])
    evidence_chunks = []
    target_entries = matches
    if rule["type"] == "signature_all":
        target_entries += [
            (k, v, p) for k, v, p in entries
            if _norm_key(k) == "detected"
            and v is True
            and ".tanda_tangan." in p
        ]

    elif rule["type"] == "signature":
        target_entries += [
            (k, v, p) for k, v, p in entries
            if _norm_key(k) == "detected"
            and v is True
            and ".tanda_tangan." in p
        ]
    if rule["type"] == "nip":
        target_entries = list(dict.fromkeys(target_entries + matches))

    for key, value, path in target_entries:
        existing = next((c for c in chunks if c.get("json_path") == path), None)
        if existing is None:
            quote = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
            existing = {
                "chunk_id": f"{Path(doc.get('path', 'document')).stem}:kv:{len(chunks)+1}",
                "text": f"{key}: {quote}", "quote": quote,
                "source_file": doc.get("path"), "json_path": path,
                "page": None, "section": None, "heading": key,
                "block_id": None, "source_id": None, "metadata": {},
                "evidence_type": "source_fragment",
            }
            chunks.append(existing)
        evidence_chunks.append(existing)

    if passed and evidence_chunks:
        unique_keys = list(dict.fromkeys(
            k for k, _, _ in target_entries
        ))

        result = {
            "status": "LOLOS",
            "evidence": [f"key:{k}" for k in unique_keys],
            "note": "Key dan nilai ditemukan"
        }
    elif rule["type"] == "signature_all" and not passed:
        result = {
            "status": "TIDAK",
            "evidence": [f"key:{k}" for k, _, _ in target_entries],
            "note": "Nama dan jabatan wajib tersedia, serta tanda_tangan atau tanda_tangan_elektronik harus true"
        }
    elif rule["type"] == "signature" and any(v is False for k, v, p in entries
                                                if _norm_key(k) == "detected" and ".tanda_tangan." in p):
        result = {"status": "TIDAK", "evidence": [f"key:{k}" for k, _, _ in matches],
                  "note": "Nama penanggung jawab ditemukan, tetapi tanda tangan tidak terdeteksi"}
    elif rule["type"] == "nip" and any(v is False for k, v, p in entries
                                       if _norm_key(k) == "adalabel" and ".nip." in p):
        result = {"status": "TIDAK", "evidence": [], "note": "Label NIP secara eksplisit tidak ditemukan"}
    elif matches or (rule["type"] in {"signature", "nip"} and target_entries):
        result = {"status": "MANUAL", "evidence": [f"key:{k}" for k, _, _ in target_entries],
                  "note": "Key ditemukan tetapi nilai/flag verifikasi belum memenuhi syarat"}
    else:
        result = {"status": "TIDAK", "evidence": [], "note": "Key wajib tidak ditemukan atau nilainya kosong"}
    return result, evidence_chunks


In [8]:
DETECTORS_BY_TYPE = KEY_RULES
NEED_ALL = set()  # Validasi multi-key ditangani eksplisit pada deterministic_key_check().

In [9]:
def normalize_patterns(value):
    """Kompatibilitas untuk metadata laporan lama; tidak dipakai untuk matching key-value."""
    if isinstance(value, str):
        return [value] if value.strip() else []
    if isinstance(value, (list, tuple)):
        return [p for p in value if isinstance(p, str) and p.strip()]
    return []

def validate_all_detectors(detector_by_type):
    """Validasi mapping field TOR dan objek key rule RAB."""
    for kind, detectors in detector_by_type.items():
        if not isinstance(detectors, dict):
            raise ValueError(f"Detector harus berupa mapping: {kind}")

        for code, rule in detectors.items():
            if kind == "TOR":
                # Aturan TOR berisi daftar kode field, misalnya ["I.A"].
                if (
                    not isinstance(rule, (list, tuple))
                    or not rule
                    or not all(isinstance(x, str) and x.strip() for x in rule)
                ):
                    raise ValueError(f"Mapping field TOR tidak valid: {kind}/{code}")
            elif not isinstance(rule, dict) or not rule.get("keys"):
                raise ValueError(f"Key rule tidak valid: {kind}/{code}")

validate_all_detectors(DETECTORS_BY_TYPE)

In [10]:
def check_item(body, kriteria_kode, kind):
    """Adapter kompatibilitas; evaluasi utama menggunakan deterministic_key_check()."""
    return {"status": "MANUAL", "evidence": [], "note": "Gunakan deterministic_key_check() berbasis JSON key-value"}

In [11]:
def validate_detector_patterns(detectors):
    """Tidak ada pola regex untuk pemeriksaan key-value."""
    return True

In [12]:
def legacy_regex_adapter(kind, doc, item):
    """Adapter lama dipertahankan agar bentuk data hasil evaluasi/report tidak berubah."""
    return deterministic_key_check(kind, doc, item)

## 6. Validator keluaran dan lokasi evidence

In [13]:
ALLOWED_STATUSES = {"LOLOS", "TIDAK", "MANUAL"}

def sentence_from_quote(quote):
    """Kalimat bukti diambil dari kutipan sumber, bukan dibuat oleh model."""
    if not quote:
        return None
    units = split_source_sentences(quote)
    return units[0] if units else quote

def validate_prediction(pred, candidates, doc_name, kind, item, method, rule_hits=None):
    status = str(pred.get("status", "MANUAL")).upper().strip()
    alasan = str(pred.get("alasan", "")).strip()
    quote = str(pred.get("evidence_quote", "") or "")
    chunk_id = pred.get("evidence_chunk_id")
    rule_hits = rule_hits or []

    if status not in ALLOWED_STATUSES:
        status, alasan = "MANUAL", "Status model tidak valid; perlu pemeriksaan manual."
    chosen = next((c for c in candidates if str(c["chunk_id"]) == str(chunk_id)), None)
    if quote:
        matching = next((c for c in candidates if c.get("quote") == quote), None)
        if chosen is None and matching is not None:
            chosen = matching
        if chosen is None or quote != chosen.get("quote"):
            status = "MANUAL"
            alasan = "Kutipan tidak identik dengan kandidat sumber; perlu pemeriksaan manual."
            chosen = None
    elif status == "LOLOS":
        status, alasan, chosen = "MANUAL", "Status LOLOS tidak memiliki kutipan evidence yang dapat diverifikasi.", None
    if status == "LOLOS" and chosen is None:
        status, alasan = "MANUAL", "Status LOLOS tidak dapat diverifikasi dengan evidence yang valid."

    loc = None
    if chosen is not None:
        start = chosen.get("quote", "").find(quote) if quote else -1
        loc = {
            "source_file": chosen.get("source_file"), "json_path": chosen.get("json_path"),
            "page": chosen.get("page"), "section": chosen.get("section"),
            "heading": chosen.get("heading"), "block_id": chosen.get("block_id"),
            "source_id": chosen.get("source_id"), "chunk_id": chosen.get("chunk_id"),
            "char_start_in_quote": start if start >= 0 else None,
            "char_end_in_quote": start + len(quote) if start >= 0 else None,
        }

    method_description = {
        "rule_based_key": "Deterministik: pencocokan key-value JSON",
        "llm_integration": "Integrasi LLM eksternal (notebook terpisah)",
        "cross_document_manual": "Pemeriksaan lintas dokumen; wajib review manusia",
        "manual": "Pemeriksaan manual",
    }.get(method, method)
    return {
        "jenis_dokumen": kind,
        "file_sumber": doc_name,
        "section_code": ROUTES[kind]["section_code"],
        "kriteria_kode": item.get("kriteria_kode"),
        "pertanyaan": item.get("pertanyaan", ""),
        "catatan": (
            ""
            if status == "LOLOS"
            else item.get("catatan_kriteria", item.get("catatan", "")) or ""
        ),
        "feedback": (
            ""
            if status == "LOLOS"
            else item.get("feedback_revisi", item.get("feedback", "")) or ""
        ),
        "metode_checklist": {"kode": method, "deskripsi": method_description,
            "rule_hit_count": len(rule_hits), "semantic_retrieval": False},
        "status": status,
        "alasan": alasan or "Belum ada alasan yang memadai; perlu pemeriksaan manual.",
        "evidence": {
            "sentence": sentence_from_quote(quote) if chosen is not None else None,
            "quote": quote if chosen is not None else None,
            "evidence_type": chosen.get("evidence_type") if chosen else None,
            "location": loc,
            "retrieval_score": chosen.get("retrieval_score") if chosen else None,
        },
        "versions": {
            "pipeline": PIPELINE_VERSION, "rules_version": RULES_VERSION,
            "rules_sha256": RULES_HASH, "model_version": model_version,
            "sft_model_version": None,
            "prompt_version": PROMPT_VERSION,
            "embedding_model": "not_used_key_value",
        },
        "review_required": status != "LOLOS" or loc is None,
    }


In [14]:
# Teks tiap sebab. Bisa diedit tanpa mengubah logika.
SEBAB_TEXT = {
    "tidak_ada":            "belum tercantum dalam dokumen",
    "tidak_terdeteksi":     "tidak terdeteksi pada dokumen",
    "nilai_kosong":         "label ada tetapi isinya kosong",
    "label_tidak_ditemukan":"isi ada tetapi label/judulnya tidak ditemukan",
}

def build_reasoning(kind, item, status, legacy_result):
    kode = item.get("kriteria_kode", "")
    base_fb = item.get("feedback_revisi") or item.get("feedback") or ""
    note = legacy_result.get("note", "")
    missing = legacy_result.get("missing_detail", [])

    if status == "LOLOS":
        return (f"Kriteria {kode} terpenuhi. {note}", "")

    if status == "TIDAK":
        if missing:
            rincian = "; ".join(
                f"{m['field']} {SEBAB_TEXT.get(m['sebab'], '')}" for m in missing
            )
            n = len(missing)
            alasan = f"Kriteria {kode} belum terpenuhi ({n} bagian kurang): {rincian}."
            # Feedback berbeda jika hanya satu bagian yang kurang
            feedback = (base_fb + f" Fokus pada: {rincian}.").strip() if n == 1 \
                       else (base_fb or f"Lengkapi {n} bagian yang kurang.")
            return (alasan, feedback)
        # Kondisi TIDAK tanpa detail field (mis. key rule RAB)
        return (f"Kriteria {kode} belum terpenuhi: {note}.", base_fb)

    # MANUAL: jelaskan apa yang perlu dicek reviewer, feedback dikosongkan
    return (f"Perlu pemeriksaan manual untuk {kode}: {note}.", "")

## Routing metode checklist TOR

- `I.A.1–I.L.1` dan `I.S.1–I.T.1`: pemeriksaan deterministik berbasis key-value JSON.
- Khusus `I.S.1`, field `I.S.1` (Tanda Tangan) dan `I.S.2` (Penanggung Jawab) diperiksa bersama.
- `I.M.1–I.R.1`: validasi keberadaan field sebelum diteruskan ke `LLM_EVALUATOR`.

Tidak ada embedding atau semantic retrieval.

## Evaluasi dokumen TOR

Kriteria deterministik diperiksa berdasarkan key JSON dan nilai terkait. Kriteria I.M.1–I.R.1 memerlukan evaluator LLM eksternal; jika belum tersedia, hasilnya `MANUAL`.

In [15]:
def evaluate_item(kind, doc_name, item):
    doc = docs_by_type[kind][doc_name]
    method = criterion_method(kind, item)
    pred = None
    rule_hits = []
    legacy_result = {"status": "MANUAL", "evidence": [], "note": "Belum dievaluasi"}
    legacy_matches = []

    if method == "rule_based_key":
        legacy_result, legacy_matches = deterministic_key_check(kind, doc, item)
        if legacy_result["status"] == "LOLOS" and legacy_matches:
            c = legacy_matches[0]
            pred = {
                "status": "LOLOS",
                "alasan": "Key JSON yang diwajibkan ditemukan dan nilainya memenuhi validasi deterministik.",
                "evidence_quote": c.get("quote", ""),
                "evidence_chunk_id": c.get("chunk_id"),
            }
        elif legacy_result["status"] == "TIDAK":
            pred = {
                "status": "TIDAK",
                "alasan": legacy_result["note"] + ".",
                "evidence_quote": "",
                "evidence_chunk_id": None,
            }
        else:
            c = legacy_matches[0] if legacy_matches else None
            pred = {
                "status": "MANUAL",
                "alasan": legacy_result["note"] + ". Perlu verifikasi reviewer.",
                "evidence_quote": c.get("quote", "") if c else "",
                "evidence_chunk_id": c.get("chunk_id") if c else None,
            }

    elif method == "llm_integration":
        # Tidak menjalankan model di notebook ini. Seluruh data dan kriteria diteruskan
        # melalui hook LLM_EVALUATOR yang berasal dari notebook LLM terpisah.
        candidates = list(doc.get("chunks", []))
        code = str(item.get("kriteria_kode", "")).strip().upper()
        is_tor_llm_criterion = kind == "TOR" and re.match(r"^I\.[M-R](?:[.\-]|$)", code)
        field_result = None
        if is_tor_llm_criterion:
            field_result = check_tor_fields(doc, code)

        if field_result is not None and field_result["status"] != "LOLOS":
            legacy_result = field_result
            pred = {
                "status": "TIDAK",
                "alasan": field_result["note"],
                "evidence_quote": "",
                "evidence_chunk_id": None,
            }
        elif field_result is not None:
            legacy_result = field_result
            try:
                pred = run_external_llm_evaluator(kind, doc_name, item, doc)
            except Exception as e:
                pred = {
                    "status": "MANUAL",
                    "alasan": f"Pipeline LLM belum tersedia/gagal dihubungkan ({type(e).__name__}).",
                    "evidence_quote": "",
                    "evidence_chunk_id": None,
                }
            if str(pred.get("status", "")).upper().strip() == "MANUAL":
                quote = field_result["quote"]
                evidence_chunk = next(
                    (candidate for candidate in candidates if candidate.get("quote") == quote),
                    None,
                )
                if evidence_chunk is None:
                    evidence_chunk = {
                        "chunk_id": f"{Path(doc.get('path', doc_name)).stem}:field:{code}",
                        "text": quote,
                        "quote": quote,
                        "source_file": doc.get("path", doc_name),
                        "json_path": "$.fields." + ", $.fields.".join(field_result["field_codes"]),
                        "page": None,
                        "section": None,
                        "heading": code,
                        "block_id": None,
                        "source_id": None,
                        "metadata": {},
                        "evidence_type": "source_fragment",
                    }
                    candidates.append(evidence_chunk)
                pred["evidence_quote"] = quote
                pred["evidence_chunk_id"] = evidence_chunk["chunk_id"]
        else:
            try:
                pred = run_external_llm_evaluator(kind, doc_name, item, doc)
            except Exception as e:
                pred = {
                    "status": "MANUAL",
                    "alasan": f"Pipeline LLM belum tersedia/gagal dihubungkan ({type(e).__name__}).",
                    "evidence_quote": "",
                    "evidence_chunk_id": None,
                }


    else:
        # Untuk metode manual pada dokumen tunggal TOR atau RAB.
        candidates = list(doc.get("chunks", []))
        pred = {
            "status": "MANUAL",
            "alasan": "Kriteria memerlukan pemeriksaan dan validasi reviewer.",
            "evidence_quote": "",
            "evidence_chunk_id": None,
        }

    if method == "rule_based_key":
        candidates = list(doc.get("chunks", []))
        if kind == "TOR":
            for candidate in legacy_matches:
                if not any(
                    existing.get("quote") == candidate.get("quote")
                    for existing in candidates
                ):
                    candidates.append(candidate)
        elif method == "llm_integration":
            candidates = list(doc.get("chunks", []))
        else:
            candidates = list(doc.get("chunks", []))

    row = validate_prediction(pred, candidates, doc_name, kind, item, method, rule_hits)

    if method in ("rule_based_key", "llm_integration"):
        row["alasan"], row["feedback"] = build_reasoning(kind, item, row["status"], legacy_result)
        if row["status"] == "LOLOS":
            row["catatan"] = ""

        # Jika LLM memberi alasan untuk MANUAL, pakai alasan LLM
    if method == "llm_integration" and row["status"] == "MANUAL" and pred.get("alasan"):
        row["alasan"] = pred["alasan"]

    if row.get("status") != "LOLOS" and not row.get("catatan"):
        row["catatan"] = legacy_result.get("note", "")

    if kind == "TOR" and method == "llm_integration" and row.get("status") == "MANUAL":
        row["feedback"] = ""

    row["evidence_pola"] = list(legacy_result.get("evidence", []))
    row["candidate_evidence"] = [{
        "chunk_id": c["chunk_id"], "quote": c["quote"], "text": c["text"],
        "score": None, "json_path": c["json_path"],
        "page": c.get("page"), "section": c.get("section"), "heading": c.get("heading"),
        "block_id": c.get("block_id"),
    } for c in candidates]
    row["rule_hits"] = []
    row["legacy_regex_result"] = {
        "status": legacy_result.get("status"),
        "matched_patterns": legacy_result.get("evidence", []),
        "note": legacy_result.get("note"),
    }
    row["legacy_regex_evidence_chunks"] = [
        {"chunk_id": c.get("chunk_id"), "patterns": legacy_result.get("evidence", [])}
        for c in legacy_matches
    ]
    row["evidence_quote"] = pred.get("evidence_quote", "")
    return row

all_results = {}
for kind, documents in docs_by_type.items():
    all_results[kind] = {}
    if not documents:
        print(f"SKIP evaluasi [{kind}]: tidak ada dokumen.")
        continue
    for doc_name in documents:
        rows = [evaluate_item(kind, doc_name, item) for item in ROUTES[kind]["rules"]["items"]]
        all_results[kind][doc_name] = rows
        counts = {s: sum(r["status"] == s for r in rows) for s in ALLOWED_STATUSES}
        print(f"Evaluated [{kind}] {doc_name}: {counts}")


Evaluated [TOR] (PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx.json: {'TIDAK': 1, 'LOLOS': 14, 'MANUAL': 9}
Evaluated [TOR] (PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx_vlm.json: {'TIDAK': 1, 'LOLOS': 14, 'MANUAL': 9}
Evaluated [TOR] (PA.2027 PNBP) FAE.001 - TOR Pemantauan dan Evaluasi Kebijakan Teknologi Pemerintah Digital Rev180826.docx.json: {'TIDAK': 11, 'LOLOS': 6, 'MANUAL': 7}
Evaluated [TOR] 0. TOR_7437.BAH.U00 Pemantauan, Pengukuran, dan Penanganan Gangguan Frekuensi Radio Dalam Kota.json: {'TIDAK': 2, 'LOLOS': 13, 'MANUAL': 9}
Evaluated [TOR] 009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran.json: {'TIDAK': 1, 'LOLOS': 14, 'MANUAL': 9}
Evaluated [TOR] 009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran_vlm.json: {'TIDAK': 1, 'LOLOS': 14, 'MANUAL': 9}
Evaluated [TOR] 01. PAA - KAK Rancangan Revisi UU KIP No. 14 Tahun 2008.json: {'TIDAK': 1, 'LOLOS': 14, 'MANUAL': 9}
Evaluated [TOR] 011 - TOR 2027 

## 10. Simpan laporan JSON dan Markdown

In [16]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

def perbaiki_teks(s):
    """Perbaiki teks yang salah encoding (UTF-8 terbaca sebagai cp1252) dan ganti tanda strip panjang dengan '-'."""
    if not isinstance(s, str):
        return s
    try:
        s = s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    return s.replace("—", "-").replace("–", "-")

def md_escape(value):
    s = perbaiki_teks(str(value if value is not None else ""))
    return (s.replace("\\", "\\\\").replace("$", "\\$").replace("|", "\\|")
             .replace("\n", " ").replace("\r", " "))

def md_table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    for r in rows:
        out.append("| " + " | ".join(md_escape(c) for c in r) + " |")
    return out

def loc_str(ev):
    loc = ev.get("location") or {}
    return "; ".join(f"{k}={v}" for k, v in loc.items() if v not in (None, ""))

def save_report(kind, doc_name, rows):
    base = f"{kind}__{Path(doc_name).stem}__checklist_report"
    kind_out_dir = OUT_DIR / kind
    kind_out_dir.mkdir(parents=True, exist_ok=True)

    counts = {s: sum(r["status"] == s for r in rows) for s in ALLOWED_STATUSES}
    section_codes = list(dict.fromkeys(r.get("section_code") for r in rows if r.get("section_code")))

    def sec_title(code):
        return perbaiki_teks(rules_by_code.get(code, {}).get("section_title", ""))

    payload = {
        "jenis_dokumen": kind,
        "file_sumber": docs_by_type[kind][doc_name]["path"],
        "section_codes": section_codes,
        "summary": {**counts, "total": len(rows)},
        "versions": {
            "pipeline": PIPELINE_VERSION, "rules_version": RULES_VERSION,
            "rules_sha256": RULES_HASH, "model_version": model_version,
            "prompt_version": PROMPT_VERSION, "embedding_model": "not_used_key_value",
        },
        "hasil": rows,
    }
    (kind_out_dir / f"{base}.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    lines = [
        f"# Checklist: {doc_name}", "",
        f"- Jenis dokumen: **{kind}**",
        f"- Ringkasan: LOLOS {counts['LOLOS']}, TIDAK {counts['TIDAK']}, MANUAL {counts['MANUAL']}, total {len(rows)}",
        f"- Pipeline: `{PIPELINE_VERSION}`",
        f"- Rules version: `{RULES_VERSION}` (`{RULES_HASH}`)",
        f"- Model version: `{model_version}`", "",
    ]

    for code in section_codes:
        sec_rows = [r for r in rows if r.get("section_code") == code]
        sc = {s: sum(r["status"] == s for r in sec_rows) for s in ALLOWED_STATUSES}
        lines += [f"## Section {code} â€” {sec_title(code)}", "",
                  f"Ringkasan: LOLOS {sc['LOLOS']}, TIDAK {sc['TIDAK']}, MANUAL {sc['MANUAL']}", ""]

        if code == "IV":
            adakah_tidak_lolos = any(r["status"] != "LOLOS" for r in sec_rows)

            if adakah_tidak_lolos:
                headers = ["Kriteria", "Status", "Pertanyaan", "Feedback", "Alasan", "Lokasi"]
                body = [[
                    r.get("kriteria_kode"),
                    r["status"],
                    r.get("pertanyaan", "") if r["status"] != "LOLOS" else "",
                    r.get("feedback", "") if r["status"] != "LOLOS" else "",
                    r.get("alasan"),
                    loc_str(r.get("evidence", {})),
                ] for r in sec_rows]
            else:
                headers = ["Kriteria", "Status", "Alasan", "Lokasi"]
                body = [[
                    r.get("kriteria_kode"),
                    r["status"],
                    r.get("alasan"),
                    loc_str(r.get("evidence", {})),
                ] for r in sec_rows]
        else:
            headers = ["Kriteria", "Pertanyaan", "Status", "Metode", "Evidence pola",
                       "Catatan", "Feedback", "Alasan", "Kalimat bukti", "Lokasi"]
            body = [[
                r.get("kriteria_kode"),
                r.get("pertanyaan"),
                r["status"],
                (r.get("metode_checklist") or {}).get("deskripsi", ""),
                ", ".join(r.get("evidence_pola", [])),
                r.get("catatan"),
                r.get("feedback"),
                r.get("alasan"),
                (r.get("evidence") or {}).get("sentence"),
                loc_str(r.get("evidence", {})),
            ] for r in sec_rows]

        lines += md_table(headers, body) + [""]

        
        for r in sec_rows:
            if r.get("detail_rows"):
                lines += [f"### Detail {r['kriteria_kode']}", ""]
                lines += md_table(r["detail_headers"], r["detail_rows"]) + [""]
            if r.get("summary_rows"):
                lines += [f"### Ringkasan {r['kriteria_kode']}", ""]
                lines += md_table(r["summary_headers"], r["summary_rows"]) + [""]

    (kind_out_dir / f"{base}.md").write_text("\n".join(lines), encoding="utf-8")
    print("Saved:", base)

saved = 0
for kind, documents in all_results.items():
    for doc_name, rows in documents.items():
        save_report(kind, doc_name, rows)
        saved += 1
print(f"Selesai. Laporan dibuat untuk {saved} dokumen di {OUT_DIR.resolve()}")
if saved == 0:
    print("Tidak ada laporan karena tidak ada dokumen yang berhasil dimuat.")

Saved: TOR__(PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx__checklist_report
Saved: TOR__(PA.2027 PNBP) AAG-001 - TOR PM Pendekatan Kota Cerdas 2027 Rev180826.docx_vlm__checklist_report
Saved: TOR__(PA.2027 PNBP) FAE.001 - TOR Pemantauan dan Evaluasi Kebijakan Teknologi Pemerintah Digital Rev180826.docx__checklist_report
Saved: TOR__0. TOR_7437.BAH.U00 Pemantauan, Pengukuran, dan Penanganan Gangguan Frekuensi Radio Dalam Kota__checklist_report
Saved: TOR__009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran__checklist_report
Saved: TOR__009 - TOR 2027 OM Sistem Perizinan Telekomunikasi Pemutakhiran_vlm__checklist_report
Saved: TOR__01. PAA - KAK Rancangan Revisi UU KIP No. 14 Tahun 2008__checklist_report
Saved: TOR__011 - TOR 2027 OM Sistem Informasi Manajemen Layanan Publik Bidang Penyiaran Pemutakhiran__checklist_report
Saved: TOR__011 - TOR 2027 OM Sistem Informasi Manajemen Layanan Publik Bidang Penyiaran Pemutakhiran_vlm__checklist_report
Save

In [17]:
# Diagnostik ringkas key rules per jenis dokumen
for kind, rules in KEY_RULES.items():
    if kind == "TOR":
        print(kind, rules)  # Mapping TOR berisi daftar kode field
        continue
    print(kind, {code: rule["keys"] for code, rule in rules.items()})

TOR {'I.A.1': ['I.A'], 'I.B.1': ['I.B'], 'I.C.1': ['I.C'], 'I.D.1': ['I.D'], 'I.E.1': ['I.E'], 'I.F.1': ['I.F'], 'I.G.1': ['I.G'], 'I.H.1': ['I.H'], 'I.I.1': ['I.I'], 'I.J.1': ['I.J'], 'I.K.1': ['I.K'], 'I.L.1': ['I.L'], 'I.M.1': ['I.M.1', 'I.M.2'], 'I.M.2': ['I.M.1'], 'I.M.3': ['I.M.2'], 'I.N.1': ['I.N'], 'I.O.1': ['I.O.1', 'I.O.2'], 'I.O.2': ['I.O.1'], 'I.O.3': ['I.O.2'], 'I.P.1': ['I.P'], 'I.Q.1': ['I.Q'], 'I.R.1': ['I.R'], 'I.S.1': ['I.S.1', 'I.S.2'], 'I.T.1': ['I.T']}


## Evaluasi dan penggunaan aman

- `LOLOS`: key wajib ditemukan dan nilai/flag memenuhi validasi.
- `TIDAK`: key wajib tidak ada/kosong atau flag menyatakan syarat tidak terpenuhi.
- `MANUAL`: data ambigu atau verifikasi/referensi belum memadai.
- Kriteria TOR I.M.1–I.R.1 memerlukan evaluator LLM eksternal melalui `LLM_EVALUATOR`.


## 12. Rekap evaluasi gabungan (markdown dinamis)

Cell ini menggabungkan hasil evaluasi semua file input ke dalam file markdown `REKAP_EVALUASI.md` dan excel `REKAP_EVALUASI.xlsx` di `hasil_cek`. State disimpan di `rekap_evaluasi_state.json` sehingga rekap bersifat dinamis antar-run:

- File input baru akan **ditambahkan** ke rekap.
- File dengan nama sama yang dievaluasi ulang akan **diperbarui** hasilnya (tidak duplikat).

Isi rekap: jumlah LOLOS/MANUAL/TIDAK per file beserta daftar kriteria (field) pada setiap status.

In [18]:
import pandas as pd
from datetime import datetime

STATUS_ORDER = ["LOLOS", "MANUAL", "TIDAK"]
SEC_ORDER = ["I"]
REKAP_STATE_PATH = OUT_DIR / "rekap_evaluasi_state.json"
REKAP_MD_PATH = OUT_DIR / "REKAP_EVALUASI.md"
REKAP_EXCEL_PATH = OUT_DIR / "REKAP_EVALUASI.xlsx"

# 1) Muat state. Entri format lama (tanpa "per_section") dibuang agar tidak salah tampil.
rekap_state = {}
if REKAP_STATE_PATH.exists():
    try:
        raw_state = json.loads(REKAP_STATE_PATH.read_text(encoding="utf-8"))
        rekap_state = {k: v for k, v in raw_state.items() if "per_section" in v}
    except (json.JSONDecodeError, OSError):
        rekap_state = {}

# 2) Merge hasil run ini, dihitung per section.
for kind, documents in all_results.items():
    for doc_name, rows in documents.items():
        per_section, fields = {}, {}
        for r in rows:
            sec = r.get("section_code") or ROUTES[kind]["section_code"]
            st = r["status"]
            per_section.setdefault(sec, {s: 0 for s in STATUS_ORDER})[st] += 1
            fields.setdefault(sec, {s: [] for s in STATUS_ORDER})[st].append(r.get("kriteria_kode", ""))
        rekap_state[f"{kind}::{doc_name}"] = {
            "jenis_dokumen": kind,
            "nama_file": doc_name,
            "total": len(rows),
            "per_section": per_section,
            "fields": fields,
            "terakhir_dievaluasi": datetime.now().isoformat(timespec="seconds"),
        }

REKAP_STATE_PATH.write_text(json.dumps(rekap_state, ensure_ascii=False, indent=2), encoding="utf-8")

# 3) Susun data rekap.
entries = sorted(rekap_state.values(), key=lambda e: (e["jenis_dokumen"], e["nama_file"]))
seen = {sec for e in entries for sec in e["per_section"]}
all_secs = [s for s in SEC_ORDER if s in seen] + sorted(seen - set(SEC_ORDER))

grand = {sec: {s: sum(e["per_section"].get(sec, {}).get(s, 0) for e in entries)
               for s in STATUS_ORDER} for sec in all_secs}

# 4) Markdown
hdr = (["No", "Jenis", "Nama file"]
       + [f"{sec} {s}" for sec in all_secs for s in STATUS_ORDER]
       + ["Total", "Terakhir dievaluasi"])

lines = [
    "# Rekap Evaluasi Kelengkapan Dokumen", "",
    f"- Diperbarui: {datetime.now().isoformat(timespec='seconds')}",
    f"- Jumlah file: **{len(entries)}**",
]
for sec in all_secs:
    g = grand[sec]
    lines.append(f"- Section {sec}: LOLOS **{g['LOLOS']}**, MANUAL **{g['MANUAL']}**, TIDAK **{g['TIDAK']}**")

lines += ["", "## Ringkasan per file", "",
          "| " + " | ".join(hdr) + " |", "|" + "---|" * len(hdr)]
for i, e in enumerate(entries, 1):
    cells = [str(i), md_escape(e["jenis_dokumen"]), md_escape(e["nama_file"])]
    cells += [str(e["per_section"].get(sec, {}).get(s, 0)) for sec in all_secs for s in STATUS_ORDER]
    cells += [str(e.get("total", 0)), md_escape(e.get("terakhir_dievaluasi", ""))]
    lines.append("| " + " | ".join(cells) + " |")

lines += ["", "## Detail kriteria per file", ""]
for e in entries:
    lines += [f"### [{e['jenis_dokumen']}] {e['nama_file']}", ""]
    for sec in all_secs:
        if sec not in e["fields"]:
            continue
        lines.append(f"**Section {sec}**")
        for s in STATUS_ORDER:
            codes = e["fields"][sec].get(s, [])
            daftar = ", ".join(f"`{c}`" for c in codes) if codes else "-"
            lines.append(f"- {s} ({len(codes)}): {daftar}")
        lines.append("")

REKAP_MD_PATH.write_text("\n".join(lines), encoding="utf-8")

# 5) Excel: satu kolom per section dan status, plus kolom detail MANUAL/TIDAK per section.
excel_data = []
for i, e in enumerate(entries, 1):
    row = {"No": i, "Jenis": e["jenis_dokumen"], "Nama File": e["nama_file"]}
    for sec in all_secs:
        for s in STATUS_ORDER:
            row[f"{sec} {s}"] = e["per_section"].get(sec, {}).get(s, 0)
    for sec in all_secs:
        for s in ["MANUAL", "TIDAK"]:
            row[f"Detail {sec} {s}"] = ", ".join(e["fields"].get(sec, {}).get(s, []))
    row["Total"] = e.get("total", 0)
    row["Terakhir Dievaluasi"] = e.get("terakhir_dievaluasi", "")
    excel_data.append(row)

pd.DataFrame(excel_data).to_excel(REKAP_EXCEL_PATH, index=False)
print(f"Rekap diperbarui: {REKAP_MD_PATH.resolve()} dan {REKAP_EXCEL_PATH.resolve()} ({len(entries)} file)")

Rekap diperbarui: D:\OneDrive\AITF\Checklist engine\hasil_cek\REKAP_EVALUASI.md dan D:\OneDrive\AITF\Checklist engine\hasil_cek\REKAP_EVALUASI.xlsx (272 file)
